# BreastDiagAI — Nettoyage, gestion des types et audit des données (Notebook 01)

**Projet :** BreastDiagAI (Groupe `IASD29`) — classification bénin (`0`) / malin (`1`)
**Objectif :** rejouer de façon interactive et commentée l'audit réalisé par `clean_data.py` (voir `cleaning_report.log`), en ajoutant :
- un contrôle explicite des **types** et de leur conversion sans perte utile (`float64 → float32`, `diagnosis → int8`) ;
- un **audit de cohérence métier** (valeurs physiquement impossibles, variables standardisées vs à échelle réelle) ;
- une **vérification que le fichier nettoyé** `breast_cancer_cleaned.csv` correspond bien au brut.

> Ce notebook **ne modifie aucun fichier du projet** : il lit les CSV et n'écrit rien sur disque.

### Plan
1. Configuration et chargement
2. Audit structurel (dimensions, colonnes, types, mémoire)
3. Valeurs manquantes et doublons
4. Gestion des types (cible, downcasting, perte de précision)
5. Cohérence métier des valeurs
6. Outliers (IQR & Z-score) — comparaison avec `cleaning_report.log`
7. Redondance (corrélations > 0.95) et équilibre des classes
8. Contrôle du fichier nettoyé
9. Synthèse

## 1. Configuration et chargement

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

TARGET_COL = "diagnosis"
RAW_NAME = "breast_cancer_40_features_1M.csv"
CLEAN_NAME = "breast_cancer_cleaned.csv"


def find_project_root() -> Path:
    """Cherche le dossier contenant le CSV brut : variable d'environnement BREASTDIAG_ROOT,
    puis dossier courant et ses parents, puis dossier frère 'fdd' (copie locale du projet)."""
    candidates = []
    if os.environ.get("BREASTDIAG_ROOT"):
        candidates.append(Path(os.environ["BREASTDIAG_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates += [cwd, *cwd.parents]
    candidates += [p / "fdd" for p in [cwd, *cwd.parents]]
    candidates += [p / "data" / "raw" for p in [cwd, *cwd.parents]]
    for c in candidates:
        if (c / RAW_NAME).exists():
            return c
    raise FileNotFoundError(f"{RAW_NAME} introuvable : définir BREASTDIAG_ROOT=<dossier du CSV>")


ROOT = find_project_root()
RAW_PATH = ROOT / RAW_NAME
CLEAN_PATH = ROOT / CLEAN_NAME
print(f"Dossier des données : {ROOT}")
print(f"CSV brut    : {RAW_PATH.name} ({RAW_PATH.stat().st_size / 1024**2:.1f} Mo)")
if CLEAN_PATH.exists():
    print(f"CSV nettoyé : {CLEAN_PATH.name} ({CLEAN_PATH.stat().st_size / 1024**2:.1f} Mo)")

Dossier des données : /home/walid/Bureau/khebich/fdd
CSV brut    : breast_cancer_40_features_1M.csv (369.8 Mo)
CSV nettoyé : breast_cancer_cleaned.csv (203.8 Mo)


In [2]:
df_raw = pd.read_csv(RAW_PATH)
print(f"Chargé : {df_raw.shape[0]:,} lignes x {df_raw.shape[1]} colonnes")
df_raw.head(3)

Chargé : 500,000 lignes x 41 colonnes


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,cell_density,nucleus_size,nucleus_texture,mitosis_rate,tumor_border_irregularity,chromatin_density,cell_variation,nuclear_variation,cytoplasm_ratio,cell_clump_thickness,diagnosis
0,16.021426,15.900191,80.338711,806.403104,-0.234153,-0.234137,1.579213,0.767435,-0.469474,0.542560,-0.463418,-0.465730,0.241962,-1.913280,-1.724918,-0.562288,-1.012831,0.314247,-0.908024,-1.412304,17.013208,-0.225776,0.067528,1141.097712,-0.544383,0.110923,-1.150994,0.375698,-0.600639,-0.291694,48.232473,9.387159,-0.013497,2.903146,0.574452,-1.220844,0.208864,-1.959670,-1.328186,0.196861,1
1,14.074327,14.990334,72.827789,622.307690,-1.478522,-0.719844,-0.460639,1.057122,0.343618,-1.763040,0.324084,-0.385082,-0.676922,0.611676,1.031000,0.931280,-0.839218,-0.309212,0.331263,0.975545,16.210277,-0.185659,-1.106335,882.935311,0.812526,1.356240,-0.072010,1.003533,0.361636,-0.645120,28.167857,9.823466,-0.035826,4.392747,0.189660,0.821903,0.087047,-0.299007,0.091761,-1.987569,0
2,14.893529,17.008014,89.946616,696.859332,-0.808494,-0.501757,0.915402,0.328751,-0.529760,0.513267,0.097078,0.968645,-0.702053,-0.327662,-0.392108,-1.463515,0.296120,0.261055,0.005113,-0.234587,17.706792,-0.420645,-0.342715,849.617165,-0.161286,0.404051,1.886186,0.174578,0.257550,-0.074446,61.816547,6.934019,0.060230,1.160965,0.681766,0.301547,-0.034712,-1.168678,1.142823,0.751933,0


## 2. Audit structurel
Le nom du fichier (`_1M`) suggère 1 million de lignes : on vérifie la taille réelle, puis on regroupe les 41 colonnes par famille.

In [3]:
print(f"Lignes : {len(df_raw):,}  |  Colonnes : {df_raw.shape[1]}")
print(f"Mémoire : {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} Mo")
print("\nTypes de données :")
print(df_raw.dtypes.value_counts().to_string())

features = [c for c in df_raw.columns if c != TARGET_COL]
families = {
    "_mean (moyenne)": [c for c in features if c.endswith("_mean")],
    "_se (erreur standard)": [c for c in features if c.endswith("_se")],
    "_worst (valeur extrême)": [c for c in features if c.endswith("_worst")],
    "biologiques (ajoutées)": [c for c in features if not c.endswith(("_mean", "_se", "_worst"))],
}
for k, v in families.items():
    print(f"\n{k} — {len(v)} variables :\n  {', '.join(v)}")

Lignes : 500,000  |  Colonnes : 41
Mémoire : 156.40 Mo

Types de données :
float64    40
int64       1

_mean (moyenne) — 10 variables :
  radius_mean, texture_mean, perimeter_mean, area_mean, smoothness_mean, compactness_mean, concavity_mean, concave_points_mean, symmetry_mean, fractal_dimension_mean

_se (erreur standard) — 10 variables :
  radius_se, texture_se, perimeter_se, area_se, smoothness_se, compactness_se, concavity_se, concave_points_se, symmetry_se, fractal_dimension_se

_worst (valeur extrême) — 10 variables :
  radius_worst, texture_worst, perimeter_worst, area_worst, smoothness_worst, compactness_worst, concavity_worst, concave_points_worst, symmetry_worst, fractal_dimension_worst

biologiques (ajoutées) — 10 variables :
  cell_density, nucleus_size, nucleus_texture, mitosis_rate, tumor_border_irregularity, chromatin_density, cell_variation, nuclear_variation, cytoplasm_ratio, cell_clump_thickness


## 3. Valeurs manquantes et doublons

In [4]:
na = df_raw.isna().sum()
print(f"Valeurs manquantes totales : {int(na.sum())}")
print(f"Colonnes avec NaN : {list(na[na > 0].index) or 'aucune'}")
inf_count = np.isinf(df_raw.select_dtypes(include=np.number)).sum().sum()
print(f"Valeurs infinies : {int(inf_count)}")

n_dup = df_raw.duplicated().sum()
n_dup_feat = df_raw.duplicated(subset=features).sum()
print(f"\nDoublons exacts (41 colonnes) : {n_dup}")
print(f"Doublons sur les 40 features (cible ignorée) : {n_dup_feat}")

Valeurs manquantes totales : 0
Colonnes avec NaN : aucune
Valeurs infinies : 0



Doublons exacts (41 colonnes) : 0
Doublons sur les 40 features (cible ignorée) : 0


**Constat :** aucune valeur manquante, infinie ou dupliquée, ce qui confirme `cleaning_report.log`. Le pipeline de prétraitement conserve un `SimpleImputer(median)` à titre **préventif**, pour les futures données saisies dans l'application.

## 4. Gestion des types
### 4.1 Variable cible
`diagnosis` est codée en `int64`. On vérifie qu'elle ne contient que `{0, 1}`, puis on la convertit en `int8` (1 octet).

In [5]:
print("Valeurs uniques de diagnosis :", sorted(int(v) for v in df_raw[TARGET_COL].unique()))
assert set(df_raw[TARGET_COL].unique()) <= {0, 1}, "Valeurs inattendues dans la cible"

df = df_raw.copy()
df[TARGET_COL] = df[TARGET_COL].astype("int8")
print("Type de diagnosis après conversion :", df[TARGET_COL].dtype)

Valeurs uniques de diagnosis : [0, 1]
Type de diagnosis après conversion : int8


### 4.2 Downcasting des 40 features `float64 → float32`
`clean_data.py` applique `pd.to_numeric(downcast="float")`. Ici, on mesure **la perte de précision** induite, pour justifier que le downcast est sans conséquence.

In [6]:
before = df.memory_usage(deep=True).sum() / 1024**2
f32 = df[features].astype("float32")

abs_err = (f32.astype("float64") - df[features]).abs()
rel_err = abs_err / df[features].abs().replace(0, np.nan)
prec = pd.DataFrame({
    "erreur_abs_max": abs_err.max(),
    "erreur_rel_max": rel_err.max(),
}).sort_values("erreur_abs_max", ascending=False)
print("Perte de précision maximale (top 5 en erreur absolue) :")
display(prec.head(5))
print(f"Erreur relative maximale sur toutes les features : {prec['erreur_rel_max'].max():.2e}")

df[features] = f32
after = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nMémoire : {before:.2f} Mo -> {after:.2f} Mo (-{(1 - after / before) * 100:.1f} %)")
print(df.dtypes.value_counts().to_string())

Perte de précision maximale (top 5 en erreur absolue) :


,erreur_abs_max,erreur_rel_max
area_worst,0.000122,5.946254e-08
area_mean,0.000121,5.947198e-08
perimeter_mean,0.000008,5.951582e-08
cell_density,0.000004,5.955082e-08
texture_mean,0.000002,5.941473e-08


Erreur relative maximale sur toutes les features : 5.96e-08

Mémoire : 153.06 Mo -> 76.77 Mo (-49.8 %)
float32    40
int8        1


**Interprétation :** l'erreur relative maximale est d'environ 6 × 10⁻⁸ (précision du float32, ~7 chiffres significatifs), bien en dessous de la précision d'une mesure morphologique. Le downcast divise la mémoire par deux environ, sans perte utile. Le point de départ (153,06 Mo) est un peu plus bas que les 156,40 Mo du log, car la cible a déjà été convertie en `int8`. L'état final (76,77 Mo) est identique à celui de `cleaning_report.log`.

## 5. Cohérence métier des valeurs
Les 41 colonnes ne sont pas toutes à la même échelle. Certaines sont des **mesures physiques** (rayon, aire, périmètre…), qui doivent être strictement positives. D'autres sont **centrées-réduites** (moyenne ≈ 0, écart-type ≈ 1), ce qui caractérise des données synthétiques ou déjà standardisées.

In [7]:
desc = df[features].describe().T[["mean", "std", "min", "max"]]
desc["standardisée"] = (desc["mean"].abs() < 0.05) & ((desc["std"] - 1).abs() < 0.05)
std_cols = desc.index[desc["standardisée"]].tolist()
real_cols = desc.index[~desc["standardisée"]].tolist()
print(f"Variables centrées-réduites (|moy|<0.05 et |std-1|<0.05) : {len(std_cols)}")
print(f"Variables à échelle réelle : {len(real_cols)} -> {real_cols}")
display(desc.loc[real_cols].round(3))

Variables centrées-réduites (|moy|<0.05 et |std-1|<0.05) : 30
Variables à échelle réelle : 10 -> ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'radius_worst', 'area_worst', 'cell_density', 'nucleus_size', 'mitosis_rate', 'tumor_border_irregularity']


,mean,std,min,max,standardisée
radius_mean,13.998,3.003,0.168,28.689,False
texture_mean,19.998,5.001,0.024,43.231,False
perimeter_mean,83.975,19.362,0.909,179.970,False
area_mean,643.922,267.268,0.088,2585.669,False
radius_worst,16.015,3.154,0.953,31.157,False
area_worst,837.231,374.642,0.130,4010.403,False
cell_density,49.984,14.987,0.001,122.558,False
nucleus_size,10.001,2.993,0.003,23.976,False
mitosis_rate,4.034,1.931,0.000,13.583,False
tumor_border_irregularity,0.500,0.198,0.000,1.398,False


In [8]:
# Les mesures physiques à échelle réelle ne doivent jamais être <= 0
neg = {c: int((df[c] <= 0).sum()) for c in real_cols}
neg = {c: n for c, n in neg.items() if n > 0}
print("Valeurs <= 0 sur les variables à échelle réelle :", neg if neg else "aucune")

Valeurs <= 0 sur les variables à échelle réelle : aucune


**Constat :** les variables à échelle réelle (rayons, aires, périmètre, et les variables biologiques comme `nucleus_size` ou `mitosis_rate`) sont cohérentes. Les autres sont centrées-réduites : leurs valeurs négatives sont normales et ne doivent **pas** être traitées comme des erreurs. Les minimums très proches de 0 sur certaines aires et certains rayons sont physiquement plausibles mais atypiques ; ils font partie des outliers de la section 6.

## 6. Outliers (IQR & Z-score)
Même méthode que `clean_data.py` : IQR (1,5 × écart interquartile) et |Z| > 4. Les outliers sont **signalés et non supprimés** : en médecine, une valeur extrême peut être un vrai cas rare (tumeur très volumineuse).

In [9]:
rows = []
for col in features:
    x = df_raw[col]
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    n_iqr = int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum())
    n_z = int((np.abs(stats.zscore(x)) > 4).sum())
    rows.append((col, n_iqr, round(n_iqr / len(x) * 100, 2), n_z))
outliers = pd.DataFrame(rows, columns=["colonne", "outliers_IQR", "%_IQR", "outliers_Z>4"]) \
    .sort_values("outliers_IQR", ascending=False).reset_index(drop=True)
display(outliers.head(10))
print(f"Total de cellules signalées (IQR) : {outliers['outliers_IQR'].sum():,}")

,colonne,outliers_IQR,%_IQR,outliers_Z>4
0,area_worst,8894,1.78,852
1,area_mean,6612,1.32,449
2,perimeter_mean,3990,0.80,82
3,concavity_se,3599,0.72,22
4,smoothness_mean,3592,0.72,25
5,cell_variation,3588,0.72,27
6,concave_points_worst,3566,0.71,33
7,texture_worst,3553,0.71,31
8,texture_mean,3547,0.71,14
9,nucleus_size,3543,0.71,17


Total de cellules signalées (IQR) : 145,136


In [10]:
# Comparaison avec les valeurs publiées dans cleaning_report.log
log_ref = {"area_worst": (8894, 852), "area_mean": (6612, 449), "perimeter_mean": (3990, 82),
           "mitosis_rate": (1772, 25), "tumor_border_irregularity": (1701, 17)}
chk = outliers.set_index("colonne").loc[list(log_ref)]
chk["log_IQR"] = [v[0] for v in log_ref.values()]
chk["log_Z"] = [v[1] for v in log_ref.values()]
chk["identique"] = (chk["outliers_IQR"] == chk["log_IQR"]) & (chk["outliers_Z>4"] == chk["log_Z"])
display(chk[["outliers_IQR", "log_IQR", "outliers_Z>4", "log_Z", "identique"]])

,outliers_IQR,log_IQR,outliers_Z>4,log_Z,identique
colonne,,,,,
area_worst,8894,8894,852,852,True
area_mean,6612,6612,449,449,True
perimeter_mean,3990,3990,82,82,True
mitosis_rate,1772,1772,25,25,True
tumor_border_irregularity,1701,1701,17,17,True


In [11]:
# Taux de malignité parmi les outliers d'area_worst : outlier = erreur ou cas clinique ?
x = df_raw["area_worst"]
q1, q3 = x.quantile([0.25, 0.75]); iqr = q3 - q1
haut = x > q3 + 1.5 * iqr
bas = x < q1 - 1.5 * iqr
print(f"area_worst : {haut.sum():,} outliers hauts, {bas.sum():,} outliers bas")
print(f"Taux de malins — global : {df_raw[TARGET_COL].mean()*100:.1f} %")
print(f"Taux de malins — outliers hauts : {df_raw.loc[haut, TARGET_COL].mean()*100:.1f} %")
if bas.sum():
    print(f"Taux de malins — outliers bas : {df_raw.loc[bas, TARGET_COL].mean()*100:.1f} %")

area_worst : 8,894 outliers hauts, 0 outliers bas
Taux de malins — global : 35.0 %
Taux de malins — outliers hauts : 100.0 %


**Interprétation :** les outliers hauts d'`area_worst` ont un taux de malignité très différent du taux global. Ils portent donc une **information diagnostique**, ce qui confirme la décision de ne pas les supprimer et d'utiliser `RobustScaler` au prétraitement.

## 7. Redondance et équilibre des classes

In [12]:
corr = df[features].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().loc[lambda s: s > 0.95].sort_values(ascending=False)
print("Paires de features avec |r| > 0.95 :")
print(pairs.round(3).to_string() if len(pairs) else "aucune")

counts = df[TARGET_COL].value_counts().sort_index()
balance = pd.DataFrame({"effectif": counts, "%": (counts / counts.sum() * 100).round(2)})
balance.index = ["Bénin (0)", "Malin (1)"]
display(balance)
print(f"Ratio majoritaire / minoritaire : {counts.max() / counts.min():.2f}")

Paires de features avec |r| > 0.95 :
radius_mean  area_mean       0.989
             radius_worst    0.952


,effectif,%
Bénin (0),325000,65.0
Malin (1),175000,35.0


Ratio majoritaire / minoritaire : 1.86


## 8. Contrôle du fichier nettoyé `breast_cancer_cleaned.csv`
Comme aucune ligne n'a été supprimée ni imputée, le fichier nettoyé doit contenir **les mêmes lignes** que le brut, à l'arrondi `float32` près. Ce contrôle sert aussi de test d'intégrité : le notebook 02 avait rencontré une copie corrompue de ce fichier.

In [13]:
if CLEAN_PATH.exists():
    df_clean = pd.read_csv(CLEAN_PATH)
    print(f"Fichier nettoyé : {df_clean.shape[0]:,} lignes x {df_clean.shape[1]} colonnes")
    same_cols = list(df_clean.columns) == list(df_raw.columns)
    print(f"Mêmes colonnes, même ordre : {same_cols}")
    print(f"NaN : {int(df_clean.isna().sum().sum())}")
    if same_cols and len(df_clean) == len(df_raw):
        max_diff = (df_clean[features] - df_raw[features]).abs().max().max()
        same_target = (df_clean[TARGET_COL].values == df_raw[TARGET_COL].values).all()
        print(f"Écart absolu max sur les features : {max_diff:.2e}")
        print(f"Cible identique ligne à ligne : {same_target}")
    del df_clean
else:
    print("breast_cancer_cleaned.csv absent : exécuter src/clean_data.py")

Fichier nettoyé : 500,000 lignes x 41 colonnes
Mêmes colonnes, même ordre : True
NaN : 0


Écart absolu max sur les features : 2.27e-04
Cible identique ligne à ligne : True


## 9. Synthèse de l'audit

In [14]:
summary = pd.DataFrame([
    ("Dimensions", f"{len(df_raw):,} lignes x {df_raw.shape[1]} colonnes (40 features + cible)"),
    ("Valeurs manquantes / infinies", f"{int(na.sum())} / {int(inf_count)}"),
    ("Doublons", f"{n_dup}"),
    ("Types", "40 float64 -> float32, diagnosis int64 -> int8"),
    ("Mémoire", f"{before:.2f} Mo -> {after:.2f} Mo"),
    ("Variables centrées-réduites", f"{len(std_cols)} / 40"),
    ("Outliers IQR max", f"{outliers.iloc[0]['colonne']} ({outliers.iloc[0]['%_IQR']} %)"),
    ("Paires redondantes > 0.95", f"{len(pairs)}"),
    ("Classes", f"{balance['%'].iloc[0]} % bénin / {balance['%'].iloc[1]} % malin"),
], columns=["Contrôle", "Résultat"])
display(summary)

,Contrôle,Résultat
0,Dimensions,"500,000 lignes x 41 colonnes (40 features + ci..."
1,Valeurs manquantes / infinies,0 / 0
2,Doublons,0
3,Types,"40 float64 -> float32, diagnosis int64 -> int8"
4,Mémoire,153.06 Mo -> 76.77 Mo
5,Variables centrées-réduites,30 / 40
6,Outliers IQR max,area_worst (1.78 %)
7,Paires redondantes > 0.95,2
8,Classes,65.0 % bénin / 35.0 % malin


### Conclusions
- Le jeu de données est **structurellement propre** : pas de valeur manquante, de doublon ou de valeur infinie, et la cible est binaire.
- La conversion de types (`float32` / `int8`) **divise la mémoire par deux environ** pour une perte de précision négligeable.
- Les valeurs extrêmes sont **conservées** : elles sont informatives sur le diagnostic, comme le montrent les outliers d'`area_worst`.
- La redondance `radius_mean` ↔ `area_mean` ↔ `radius_worst` et le déséquilibre 65/35 sont confirmés. Ils sont traités en aval par le choix du modèle et par `class_weight='balanced'`.
- La majorité des variables sont centrées-réduites, un indice fort de **données synthétiques** : prudence sur toute interprétation clinique.

**Étape suivante :** `02_statistical_eda_validation.ipynb` (EDA et tests statistiques), puis `src/preprocessing.py`.